Demonstration of connecting to the AMPEL VRO alert stream published at Hopscotch.

### 1. Obtain current AMPEL alert set (in teststream)

In [ ]:
from hop import Stream
from hop.io import StartPosition
import ampel_report_tools.AmpelReport import AmpelTransientReport
import ampel_report_tools.AmpelReportSet import AmpelReportSet

In [ ]:
host = 'kafka://kafka.scimma.org/'
topic = 'ampel.lsst.reports-test'

In [ ]:
stream = Stream(start_at=StartPosition.EARLIEST, until_eos=True)

In [ ]:
messages = []
with stream.open(host+topic, "r") as s:
    for message in s:
        messages.append( message )

In [ ]:
print(f'Found {len(messages)} alerts.') 

### 2. Load a streamed AMPEL Report and inspect the content.

In [ ]:
# Create an AmpelTransientReport based on the content of a single kafka report
v = AmpelTransientReport( messages[0].content )

In [ ]:
_ = v.plot_lightcurve()

In [ ]:
_ = v.show_classification()

### 3. Access a set of Ampel Reports 
Add filters based on lightcurve, feature, host or classification information.
Plot and react to transient passing thresholds.

In [ ]:
ars = AmpelReportSet( 
    [ 
        AmpelTransientReport(messages[k].content) 
        for k in range(10)
    ] 
)

In [ ]:
# Sample filter - minimal transient duration
ars.filter_age(min_duration=7)
ars.print_status()

In [ ]:
# Minimal TDE probability
ars.filter_class_prob( 'P(TDE)', min_prob=0.1 )
ars.print_status()

In [ ]:
# Box sky region selection 
ars.filter_sky_region(ra_range=[-50,100], dec_range=[-50,30])
ars.print_status()

In [ ]:
# Plot summary of all active transients in the set
# (host cutout bug still there)
_ = ars.plot_summary_rows()

In [ ]:
# Reset filters
ars.clear_filters()
ars.print_status()